# LowBitSparse — M0 基线 (Colab A100)
挂载 Drive → 装依赖 → 跑通 Qwen2.5-0.5B FP16 基线 (PPL / 延迟 / 显存)。

In [1]:
# 1) 确认 GPU (应显示 A100)
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-127e7605-1370-6ea5-6184-4f80ba64ea39)


In [2]:
# 2) 挂载 Google Drive (checkpoint / results 落盘, 防断连丢失)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# 3) 获取代码:git clone 你的仓库,或上传后 cd 进目录
# !git clone <your-repo-url> LowBitSparse
%cd /content/drive/MyDrive/LowBitSparse/
!ls

/content/drive/MyDrive/LowBitSparse
configs  lowbitsparse  main.py	  requirements.txt  scripts
doc	 LowBitSparse  README.MD  results	    tests


In [5]:
# 4) 安装依赖
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 9.5 MB/s eta 0:00:00


In [6]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKE")

In [8]:
# 5) 跑基线 (完整评测)
!python main.py eval --config configs/qwen0.5b_base.yaml
!python main.py quant --config configs/qwen0.5b_int8.yaml
!python main.py quant --config configs/qwen0.5b_int4.yaml
!python main.py quant --config configs/qwen0.5b_gptq_int4.yaml
!python main.py quant --config configs/qwen0.5b_awq_int4.yaml

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 1934.81it/s]
[01:37:28] INFO lowbitsparse: 模型已加载: Qwen/Qwen2.5-0.5B-Instruct
[01:37:28] INFO lowbitsparse: 参数量 494.033M, 体积 942.3MB
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
ppl:  99% 146/147 [00:05<00:00, 24.64it/s]
[01:37:37] INFO lowbitsparse: WikiText-2 PPL = 14.2445
[01:38:01] INFO lowbitsparse: prefill 29.5ms, decode 37.7 tok/s, peak 4574.341MB
[01:38:01] INFO lowbitsparse: 结果已保存: results/m0_fp16_baseline.json
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 1787.70it/s]
[01:38:12] INFO lowbitsparse: 模型已加载: Qwen/Qwen2.5-0.5B-Instruct
[01:38:12] INFO lowbitsparse: 已量化 168 个 Linear (method=rtn, bits=8, group=128, sym=False)
[01:38:12] INFO lowbitsparse: 量化后 

In [11]:
# 6) 查看基线结果并备份到 Drive
import json, shutil, os
with open('results/m0_fp16_baseline.json') as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))
os.makedirs('/content/drive/MyDrive/LowBitSparse/results', exist_ok=True)
shutil.copy('results/m0_fp16_baseline.json', '/content/drive/MyDrive/LowBitSparse/results/')

{
  "exp_id": "m0_fp16_baseline",
  "env": {
    "timestamp": "2026-07-21T05:19:38.014934+00:00",
    "torch": "2.11.0+cu128",
    "cuda": "12.8",
    "gpu": "NVIDIA A100-SXM4-40GB"
  },
  "config": {
    "exp_id": "m0_fp16_baseline",
    "seed": 42,
    "out_dir": "results",
    "model": {
      "name": "Qwen/Qwen2.5-0.5B-Instruct",
      "dtype": "float16",
      "device": "cuda"
    },
    "eval": {
      "seqlen": 2048,
      "stride": 2048,
      "max_samples": null
    },
    "profile": {
      "prefill_len": 512,
      "decode_tokens": 128
    }
  },
  "size": {
    "params": 494032768,
    "params_millions": 494.033,
    "bytes": 988065536,
    "size_mb": 942.293
  },
  "ppl": {
    "ppl": 14.2445,
    "seqlen": 2048,
    "stride": 2048,
    "n_tokens": 299078
  },
  "latency": {
    "prefill_len": 512,
    "decode_tokens": 128,
    "prefill_ms_median": 30.42,
    "decode_tps_median": 37.19
  },
  "memory": {
    "peak_mb": 4574.341,
    "current_mb": 959.292
  }
}


SameFileError: 'results/m0_fp16_baseline.json' and '/content/drive/MyDrive/LowBitSparse/results/m0_fp16_baseline.json' are the same file

In [9]:
!pip install -q -U "datasets>=2.20,<3.0" "huggingface_hub>=0.24"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 20.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
